# 04 - Phân lớp Neural Network
## Difficulty Classification with TensorFlow/Keras

Mục tiêu:
- Phân loại độ khó: **Easy / Medium / Hard**
- Huấn luyện MLP và Deep Dense NN
- Đánh giá bằng Accuracy, F1, Confusion Matrix

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import json
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from scipy.sparse import load_npz

os.makedirs('../report/images', exist_ok=True)
os.makedirs('../models', exist_ok=True)
plt.style.use('seaborn-v0_8-darkgrid')
print(f"TensorFlow version: {tf.__version__}")

## 4.1 Load Data

In [ ]:
tfidf_matrix = load_npz('../data/processed/tfidf_matrix.npz')
df = pd.read_csv('../data/processed/unified_data.csv')

X = tfidf_matrix.toarray().astype(np.float32)
y = df['difficulty_encoded'].values.astype(np.int32)

label_names = ['Easy', 'Medium', 'Hard']
print(f"X shape: {X.shape}")
print(f"y distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

## 4.2 Train/Val/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train
)
input_dim = X_train.shape[1]
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print(f"Input dim: {input_dim}")

## 4.3 Mô hình 1: MLP

In [ ]:
def build_mlp(input_dim, n_classes=3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(n_classes, activation='softmax')
    ], name='MLP')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

mlp = build_mlp(input_dim)
mlp.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_accuracy'),
    keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5, min_lr=1e-6)
]

print("Training MLP...")
hist_mlp = mlp.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=256,
    callbacks=callbacks, verbose=1
)

In [ ]:
# Evaluate MLP
y_pred_mlp = np.argmax(mlp.predict(X_test, verbose=0), axis=1)
acc_mlp = accuracy_score(y_test, y_pred_mlp)
f1_mlp = f1_score(y_test, y_pred_mlp, average='weighted')
print(f"MLP - Accuracy: {acc_mlp:.4f}, F1: {f1_mlp:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=label_names))
mlp.save('../models/mlp_model.keras')

## 4.4 Mô hình 2: Deep Dense NN

In [ ]:
def build_deep_nn(input_dim, n_classes=3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Dropout(0.4),
        layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Dropout(0.35),
        layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(n_classes, activation='softmax')
    ], name='DeepDenseNN')
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

dnn = build_deep_nn(input_dim)
dnn.summary()

In [ ]:
print("Training Deep Dense NN...")
hist_dnn = dnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=256,
    callbacks=callbacks, verbose=1
)

y_pred_dnn = np.argmax(dnn.predict(X_test, verbose=0), axis=1)
acc_dnn = accuracy_score(y_test, y_pred_dnn)
f1_dnn = f1_score(y_test, y_pred_dnn, average='weighted')
print(f"DeepNN - Accuracy: {acc_dnn:.4f}, F1: {f1_dnn:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_dnn, target_names=label_names))
dnn.save('../models/deepnn_model.keras')

## 4.5 Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Neural Network Training History', fontsize=16, fontweight='bold')

# MLP
axes[0,0].plot(hist_mlp.history['loss'], label='Train', color='#4C72B0', lw=2)
axes[0,0].plot(hist_mlp.history['val_loss'], label='Val', color='#DD8452', lw=2, ls='--')
axes[0,0].set_title('MLP - Loss'); axes[0,0].legend()

axes[0,1].plot(hist_mlp.history['accuracy'], label='Train', color='#55A868', lw=2)
axes[0,1].plot(hist_mlp.history['val_accuracy'], label='Val', color='#C44E52', lw=2, ls='--')
axes[0,1].set_title('MLP - Accuracy'); axes[0,1].legend()

# Deep NN
axes[1,0].plot(hist_dnn.history['loss'], label='Train', color='#4C72B0', lw=2)
axes[1,0].plot(hist_dnn.history['val_loss'], label='Val', color='#DD8452', lw=2, ls='--')
axes[1,0].set_title('Deep NN - Loss'); axes[1,0].legend()

axes[1,1].plot(hist_dnn.history['accuracy'], label='Train', color='#55A868', lw=2)
axes[1,1].plot(hist_dnn.history['val_accuracy'], label='Val', color='#C44E52', lw=2, ls='--')
axes[1,1].set_title('Deep NN - Accuracy'); axes[1,1].legend()

for ax in axes.flat: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('../report/images/nb04_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.6 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')

for ax, y_pred, name in zip(axes, [y_pred_mlp, y_pred_dnn], ['MLP', 'DeepNN']):
    cm = confusion_matrix(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(f'{name}\nAccuracy={acc:.4f}, F1={f1:.4f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.tight_layout()
plt.savefig('../report/images/nb04_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 4.7 Summary

In [ ]:
best_model_name = 'MLP' if acc_mlp >= acc_dnn else 'DeepNN'
print(f"\n{'='*50}")
print(f"MODEL COMPARISON")
print(f"{'='*50}")
print(f"MLP:    Acc={acc_mlp:.4f} | F1={f1_mlp:.4f}")
print(f"DeepNN: Acc={acc_dnn:.4f} | F1={f1_dnn:.4f}")
print(f"\nBest model: {best_model_name}")

meta = {'best_model': best_model_name, 'input_dim': int(input_dim),
        'label_map': {'Easy': 0, 'Medium': 1, 'Hard': 2},
        'mlp': {'accuracy': float(acc_mlp), 'f1': float(f1_mlp)},
        'deepnn': {'accuracy': float(acc_dnn), 'f1': float(f1_dnn)}}
with open('../models/nn_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("\nModels saved to ../models/")